In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split # <--- IMPORTED HERE

# ==========================================
# 1. SETUP & LOAD
# ==========================================
print("Setting up NHIS 2020 dataset for ML Imputation...")
base_dir = '/content/drive/MyDrive/Arthritis version 2/NHIS 2020/'
file_path_2020 = os.path.join(base_dir, 'adult20.csv')

nhis_feature_mapping = {
    'AGEP_A': 'Age', 'HISPALLP_A': 'Race', 'SEX_A': 'Gender', 'RATCAT_A': 'Income level',
    'EDUC_A': 'Education level', 'MARITAL_A': 'Marital status', 'SMKEV_A': 'Smoking status',
    'DRKBNG30D_A': 'Alcohol consumption', 'MODTPR_A': 'Physical activity', 'WELLVIS_A': 'Regular checkup',
    'HOUTENURE_A': 'Housing status', 'HEARINGDF_A': 'DEAF', 'BMICAT_A': 'BMI', 'VISIONDF_A': 'BLIND',
    'COGMEMDFF_A': 'Difficulty concentrating', 'DIFF_A': 'Difficulty walking', 'HICOV_A': 'Health insurance',
    'MHTHRPY_A': 'Mental health', 'HYPMED_A': 'Blood pressure', 'CHLEV_A': 'Cholesterol awareness',
    'ARTHEV_A': 'Arthritis'
}

df_nhis_2020 = pd.read_csv(file_path_2020, usecols=list(nhis_feature_mapping.keys()))
df_nhis_2020 = df_nhis_2020.rename(columns=nhis_feature_mapping)

# ==========================================
# 2. APPLY MAPPINGS
# ==========================================
# 1. Age
df_nhis_2020['Age'] = df_nhis_2020['Age'].replace([97, 98, 99], np.nan)
age_bins = [17, 24, 29, 34, 39, 44, 49, 54, 59, 64, 69, 74, 79, 84, 150]
age_labels = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
df_nhis_2020['Age'] = pd.cut(df_nhis_2020['Age'], bins=age_bins, labels=age_labels).astype('float')

# 2. Gender
df_nhis_2020['Gender'] = df_nhis_2020['Gender'].replace({2: 0, 7: np.nan, 8: np.nan, 9: np.nan})

# 3. Race & Income Level
df_nhis_2020['Race'] = df_nhis_2020['Race'].replace([8, 9, 97, 99], np.nan)
df_nhis_2020['Income level'] = df_nhis_2020['Income level'].apply(lambda x: x if x <= 14 else np.nan)

# 4. Education level & BMI
df_nhis_2020['Education level'] = df_nhis_2020['Education level'].replace([97, 98, 99], np.nan)
df_nhis_2020['BMI'] = df_nhis_2020['BMI'].replace([9], np.nan)

# 5. Marital status
df_nhis_2020['Marital status'] = df_nhis_2020['Marital status'].replace({3: 0, 7: np.nan, 8: np.nan, 9: np.nan})

# 6. Alcohol consumption
df_nhis_2020['Alcohol consumption'] = df_nhis_2020['Alcohol consumption'].apply(
    lambda x: 0 if x == 0 else (1 if 1 <= x <= 60 else np.nan)
)

# 7. Physical activity, Regular checkup, Housing status
df_nhis_2020['Physical activity'] = df_nhis_2020['Physical activity'].replace([7, 8, 9], np.nan)
df_nhis_2020['Regular checkup'] = df_nhis_2020['Regular checkup'].replace([7, 8, 9], np.nan)
df_nhis_2020['Housing status'] = df_nhis_2020['Housing status'].replace([7, 8, 9], np.nan)

# 8. Disability Features
disability_cols = ['DEAF', 'BLIND', 'Difficulty concentrating', 'Difficulty walking']
for col in disability_cols:
    df_nhis_2020[col] = df_nhis_2020[col].replace({1: 0, 2: 1, 3: 2, 4: 3, 7: np.nan, 8: np.nan, 9: np.nan})

# 9. Binary Features
binary_cols = ['Smoking status', 'Health insurance', 'Mental health', 'Blood pressure', 'Cholesterol awareness', 'Arthritis']
for col in binary_cols:
    df_nhis_2020[col] = df_nhis_2020[col].replace({2: 0, 7: np.nan, 8: np.nan, 9: np.nan})

# Drop missing target ONLY
print("Dropping rows missing the target variable (Arthritis) ONLY...")
df_clean = df_nhis_2020.dropna(subset=['Arthritis']).reset_index(drop=True)

# ==========================================
# NEW: 2.5 ISOLATE 20% FOR TESTING
# ==========================================
print("Splitting dataset to isolate exactly 20% for testing...")
# We use stratify=df_clean['Arthritis'] to keep the exact same class balance
_, df_test = train_test_split(df_clean, test_size=0.2, stratify=df_clean['Arthritis'], random_state=42)
df_test = df_test.reset_index(drop=True)

print(f"Original Full Data Shape: {df_clean.shape}")
print(f"20% Test Set Data shape ready for ML Imputation: {df_test.shape}")

# ==========================================
# 3. PREPARE FOR NEURAL NETWORKS (Using 20% subset)
# ==========================================
X_test_2020 = df_test.drop('Arthritis', axis=1) # <--- NOW USING df_test
y_test_2020 = df_test['Arthritis']

nan_mask = X_test_2020.isna()
input_dim = X_test_2020.shape[1]

# Temporary mean imputation and scaling for Neural Nets
imputer = SimpleImputer(strategy='mean')
X_temp = imputer.fit_transform(X_test_2020)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_temp)

feature_limits = {
    'Age': (1, 14), 'Race': (1, 7), 'Gender': (0, 1), 'Income level': (1, 14),
    'Education level': (1, 10), 'Marital status': (0, 2), 'Smoking status': (0, 1),
    'Alcohol consumption': (0, 1), 'Physical activity': (0, 1), 'Regular checkup': (0, 5),
    'Housing status': (0, 6), 'DEAF': (0, 3), 'BMI': (1, 4), 'BLIND': (0, 3),
    'Difficulty concentrating': (0, 3), 'Difficulty walking': (0, 3),
    'Health insurance': (0, 1), 'Mental health': (0, 1), 'Blood pressure': (0, 1),
    'Cholesterol awareness': (0, 1)
}

def finalize_imputation(X_original, X_reconstructed):
    X_recon_inv = scaler.inverse_transform(X_reconstructed)
    X_recon_df = pd.DataFrame(X_recon_inv, columns=X_original.columns)
    X_final = X_original.copy()
    for col in X_final.columns:
        X_final.loc[nan_mask[col], col] = X_recon_df.loc[nan_mask[col], col]
    X_final = X_final.round().astype(int)
    for feature, (lower, upper) in feature_limits.items():
        if feature in X_final.columns:
            X_final[feature] = X_final[feature].clip(lower=lower, upper=upper)
    return X_final

EPOCHS = 50
BATCH_SIZE = 256

# ==========================================
# 4. AE-5 IMPUTATION
# ==========================================
print("\n[1/2] Training Autoencoder-5 (AE-5) on NHIS 2020 (20% Split)...")
ae5 = Sequential([
    Input(shape=(input_dim,)),
    Dense(16, activation='relu'),
    Dense(8, activation='relu'),
    Dense(16, activation='relu'),
    Dense(input_dim, activation='linear')
])
ae5.compile(optimizer='adam', loss='mse')
ae5.fit(X_scaled, X_scaled, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0, validation_split=0.1)

print("Reconstructing missing values with AE-5...")
X_ae5_pred = ae5.predict(X_scaled, verbose=0)
X_ae5_final = finalize_imputation(X_test_2020, X_ae5_pred)

df_ae5_test = pd.concat([X_ae5_final, y_test_2020], axis=1)
ae5_out_path = os.path.join(base_dir, 'NHIS_2020_Test_Imputed_AE5.csv')
df_ae5_test.to_csv(ae5_out_path, index=False)
print(f"✅ Saved AE-5 Imputed 2020 Test Set ({len(df_ae5_test):,} rows)")

# ==========================================
# 5. GAN IMPUTATION
# ==========================================
print("\n[2/2] Training GAN Imputer on NHIS 2020 (20% Split)... (This takes a moment)")
generator = Sequential([
    Input(shape=(input_dim,)),
    Dense(64, activation='relu'),
    Dense(128, activation='relu'),
    Dense(input_dim, activation='sigmoid')
])
discriminator = Sequential([
    Input(shape=(input_dim,)),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])
discriminator.compile(optimizer=Adam(learning_rate=0.0002), loss='binary_crossentropy')
discriminator.trainable = False

gan_input = Input(shape=(input_dim,))
generated_data = generator(gan_input)
gan_output = discriminator(generated_data)
gan = Model(gan_input, gan_output)
gan.compile(optimizer=Adam(learning_rate=0.0002), loss='binary_crossentropy')

batch_count = int(X_scaled.shape[0] / BATCH_SIZE)
GAN_EPOCHS = 100
for epoch in range(GAN_EPOCHS):
    for _ in range(batch_count):
        real_data = X_scaled[np.random.randint(0, X_scaled.shape[0], size=BATCH_SIZE)]
        noise = np.random.normal(0, 1, size=[BATCH_SIZE, input_dim])
        fake_data = generator.predict(real_data + noise * 0.1, verbose=0)

        discriminator.train_on_batch(real_data, np.ones((BATCH_SIZE, 1)))
        discriminator.train_on_batch(fake_data, np.zeros((BATCH_SIZE, 1)))

        noise = np.random.normal(0, 1, size=[BATCH_SIZE, input_dim])
        gan.train_on_batch(real_data + noise * 0.1, np.ones((BATCH_SIZE, 1)))

print("Reconstructing missing values with GAN...")
X_gan_pred = generator.predict(X_scaled, verbose=0)
X_gan_final = finalize_imputation(X_test_2020, X_gan_pred)

df_gan_test = pd.concat([X_gan_final, y_test_2020], axis=1)
gan_out_path = os.path.join(base_dir, 'NHIS_2020_Test_Imputed_GAN.csv')
df_gan_test.to_csv(gan_out_path, index=False)
print(f"✅ Saved GAN Imputed 2020 Test Set ({len(df_gan_test):,} rows)")

print("\n🎉 ALL IMPUTATIONS FINISHED SUCCESSFULLY!")

Setting up NHIS 2020 dataset for ML Imputation...
Dropping rows missing the target variable (Arthritis) ONLY...
Splitting dataset to isolate exactly 20% for testing...
Original Full Data Shape: (31525, 21)
20% Test Set Data shape ready for ML Imputation: (6305, 21)

[1/2] Training Autoencoder-5 (AE-5) on NHIS 2020 (20% Split)...
Reconstructing missing values with AE-5...
✅ Saved AE-5 Imputed 2020 Test Set (6,305 rows)

[2/2] Training GAN Imputer on NHIS 2020 (20% Split)... (This takes a moment)
Reconstructing missing values with GAN...
✅ Saved GAN Imputed 2020 Test Set (6,305 rows)

🎉 ALL IMPUTATIONS FINISHED SUCCESSFULLY!


In [ ]:
import os

gan_dir = '/content/drive/MyDrive/Arthritis version 2/NHIS 2022/GAN filling method/Saved_Models/'
ae5_dir = '/content/drive/MyDrive/Arthritis version 2/NHIS 2022/AE-5 filling method/Saved_Models/'

print("📁 GAN Models Folder Contents:")
if os.path.exists(gan_dir):
    for file in os.listdir(gan_dir):
        if file.endswith('.pkl'): print(f" - {file}")
else:
    print("Folder not found.")

print("\n📁 AE-5 Models Folder Contents:")
if os.path.exists(ae5_dir):
    for file in os.listdir(ae5_dir):
        if file.endswith('.pkl'): print(f" - {file}")
else:
    print("Folder not found.")

📁 GAN Models Folder Contents:
 - Scaler_No.pkl
 - Logistic_Regression_No.pkl
 - Random_Forest_No.pkl
 - Gradient_Boosting_No.pkl
 - XGBoost_No.pkl
 - LightGBM_No.pkl
 - Decision_Tree_No.pkl
 - AdaBoost_No.pkl
 - Stacked_Ensemble_No.pkl
 - Scaler_ROS.pkl
 - Logistic_Regression_ROS.pkl
 - Random_Forest_ROS.pkl
 - Gradient_Boosting_ROS.pkl
 - XGBoost_ROS.pkl
 - LightGBM_ROS.pkl
 - Decision_Tree_ROS.pkl
 - AdaBoost_ROS.pkl
 - Stacked_Ensemble_ROS.pkl
 - Scaler_RUS.pkl
 - Logistic_Regression_RUS.pkl
 - Random_Forest_RUS.pkl
 - Gradient_Boosting_RUS.pkl
 - XGBoost_RUS.pkl
 - LightGBM_RUS.pkl
 - Decision_Tree_RUS.pkl
 - AdaBoost_RUS.pkl
 - Stacked_Ensemble_RUS.pkl
 - Scaler_SMOTE.pkl
 - Logistic_Regression_SMOTE.pkl
 - Random_Forest_SMOTE.pkl
 - Gradient_Boosting_SMOTE.pkl
 - XGBoost_SMOTE.pkl
 - LightGBM_SMOTE.pkl
 - Decision_Tree_SMOTE.pkl
 - AdaBoost_SMOTE.pkl
 - Stacked_Ensemble_SMOTE.pkl
 - Scaler_AdaSyn.pkl
 - Logistic_Regression_AdaSyn.pkl
 - Random_Forest_AdaSyn.pkl
 - Gradient_Boosti

# **Evaluation**

In [5]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. SETUP PATHS & LOAD 2020 DATA
# ==========================================
print("Initializing Temporal Validation (Testing 2022 Top 5 Models on 2020 Data)...")

base_dir_2020 = '/content/drive/MyDrive/Arthritis version 2/NHIS 2020/'
base_dir_2022 = '/content/drive/MyDrive/Arthritis version 2/NHIS 2022/'

# Load both pristine 2020 Imputed Test Sets (Now 20% isolated sets)
print("Loading 2020 Imputed Test Sets (GAN and AE-5)...")
df_gan = pd.read_csv(os.path.join(base_dir_2020, 'NHIS_2020_Test_Imputed_GAN.csv'))
df_ae5 = pd.read_csv(os.path.join(base_dir_2020, 'NHIS_2020_Test_Imputed_AE5.csv'))

# Separate features and targets
X_test_gan = df_gan.drop('Arthritis', axis=1)
y_test_gan = df_gan['Arthritis']

X_test_ae5 = df_ae5.drop('Arthritis', axis=1)
y_test_ae5 = df_ae5['Arthritis']

# Added confirmation print statements so you can verify it's the 20% split
print(f"✅ Loaded GAN Test Set Shape: {X_test_gan.shape}")
print(f"✅ Loaded AE-5 Test Set Shape: {X_test_ae5.shape}\n")

# ==========================================
# 2. DEFINE THE TRUE TOP 5 MODELS
# ==========================================
gan_model_dir = os.path.join(base_dir_2022, 'GAN filling method', 'Saved_Models')
ae5_model_dir = os.path.join(base_dir_2022, 'Autoencoder-5 filling method', 'Saved_Models')

models_to_test = [
    {
        "name": "1. Gradient Boosting (GAN + ROS)",
        "path": os.path.join(gan_model_dir, 'Gradient_Boosting_ROS.pkl'),
        "imputer": "GAN"
    },
    {
        "name": "2. Gradient Boosting (GAN + RUS)",
        "path": os.path.join(gan_model_dir, 'Gradient_Boosting_RUS.pkl'),
        "imputer": "GAN"
    },
    {
        "name": "3. Stacked Ensemble (AE-5 + RUS)",
        "path": os.path.join(ae5_model_dir, 'Stacked_Ensemble_RUS.pkl'),
        "imputer": "AE5"
    },
    {
        "name": "4. Stacked Ensemble (GAN + RUS)",
        "path": os.path.join(gan_model_dir, 'Stacked_Ensemble_RUS.pkl'),
        "imputer": "GAN"
    },
    {
        "name": "5. AdaBoost (GAN + ROS)",
        "path": os.path.join(gan_model_dir, 'AdaBoost_ROS.pkl'),
        "imputer": "GAN"
    }
]

# ==========================================
# 3. EVALUATION LOOP
# ==========================================
print("="*75)
print(f"{'TEMPORAL VALIDATION RESULTS (NHIS 2022 Models -> NHIS 2020 Data)':^75}")
print("="*75)

results = []

for model_info in models_to_test:
    model_name = model_info['name']
    model_path = model_info['path']
    imputer_type = model_info['imputer']

    # 1. Select the correct test set based on the imputation method
    if imputer_type == "GAN":
        X_test, y_test = X_test_gan, y_test_gan
    else:
        X_test, y_test = X_test_ae5, y_test_ae5

    # 2. Check if model exists and load it
    if not os.path.exists(model_path):
        print(f"⚠️ SKIPPED: Could not find model file at -> {model_path}")
        continue

    try:
        model = joblib.load(model_path)
    except Exception as e:
        print(f"⚠️ SKIPPED: Error loading {model_name}. Details: {e}")
        continue

    # 3. Generate Predictions
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = model.predict(X_test)

    y_pred = model.predict(X_test)

    # 4. Calculate Metrics
    auroc = roc_auc_score(y_test, y_prob)
    bal_acc = balanced_accuracy_score(y_test, y_pred)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)

    # 5. Store & Print
    results.append({
        "Model": model_name,
        "Imputation_Tested_On": imputer_type,
        "AUROC": auroc,
        "Bal_Acc": bal_acc,
        "Sensitivity": sensitivity,
        "Specificity": specificity
    })

    print(f"\n✅ {model_name} (Tested on {imputer_type} Data)")
    print(f"   AUROC: {auroc:.4f} | Bal Acc: {bal_acc:.4f} | Sens: {sensitivity:.4f} | Spec: {specificity:.4f}")

# ==========================================
# 4. SAVE FINAL SUMMARY TABLE
# ==========================================
if results:
    results_df = pd.DataFrame(results)
    save_path = os.path.join(base_dir_2020, 'Temporal_Validation_True_Top5_20Percent.csv')
    results_df.to_csv(save_path, index=False)
    print("\n" + "="*75)
    print(f"🎉 Summary table saved to: {save_path}")
    print("="*75)
else:
    print("\n⚠️ No models were successfully evaluated. Please check your model file paths.")

Initializing Temporal Validation (Testing 2022 Top 5 Models on 2020 Data)...
Loading 2020 Imputed Test Sets (GAN and AE-5)...
✅ Loaded GAN Test Set Shape: (6305, 20)
✅ Loaded AE-5 Test Set Shape: (6305, 20)

     TEMPORAL VALIDATION RESULTS (NHIS 2022 Models -> NHIS 2020 Data)      

✅ 1. Gradient Boosting (GAN + ROS) (Tested on GAN Data)
   AUROC: 0.8314 | Bal Acc: 0.7456 | Sens: 0.8249 | Spec: 0.6662

✅ 2. Gradient Boosting (GAN + RUS) (Tested on GAN Data)
   AUROC: 0.8309 | Bal Acc: 0.7505 | Sens: 0.8273 | Spec: 0.6737

✅ 3. Stacked Ensemble (AE-5 + RUS) (Tested on AE5 Data)
   AUROC: 0.8337 | Bal Acc: 0.7530 | Sens: 0.7890 | Spec: 0.7171

✅ 4. Stacked Ensemble (GAN + RUS) (Tested on GAN Data)
   AUROC: 0.8310 | Bal Acc: 0.7496 | Sens: 0.8195 | Spec: 0.6797

✅ 5. AdaBoost (GAN + ROS) (Tested on GAN Data)
   AUROC: 0.8274 | Bal Acc: 0.7494 | Sens: 0.8076 | Spec: 0.6912

🎉 Summary table saved to: /content/drive/MyDrive/Arthritis version 2/NHIS 2020/Temporal_Validation_True_Top5_20Perc